# 1. Introduction
This notebook performs an initial exploratory data analysis (EDA) for the Telco Customer Churn dataset. 
The goal is to understand the data, identify potential issues, and establish a baseline for modeling.


In [2]:
# Manipulación de datos
import pandas as pd
import numpy as np
import math

# Configuración del sistema
import sys
from pathlib import Path

ROOT_DIR = Path().resolve().parent
sys.path.append(str(ROOT_DIR))


# Visualización
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno  # Visualización de valores faltantes
import matplotlib.pyplot as plt


# Estadísticas y métricas
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Modelado rápido para baseline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, recall_score, confusion_matrix, classification_report

#Comprobacion de versiones de librerías
print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")
print(f"Seaborn version: {sns.__version__}")
print(f"Scikit-learn version: {sys.modules['sklearn'].__version__}")    
print(f"Missingno version: {msno.__version__}")


Pandas version: 2.3.3
Numpy version: 2.3.5
Seaborn version: 0.13.2
Scikit-learn version: 1.8.0
Missingno version: 0.5.2


# 2. Data Loading
Load the dataset from the raw data directory and inspect basic information.


In [3]:
# Carga el dataset 
df = pd.read_csv(r"C:\Users\diego\Documents\Analisis\churn-ml-system\data\raw\WA_Fn-UseC_-Telco-Customer-Churn.csv")
# Muestra información básica del dataset
df.info()
df.describe()

df.head().style.set_table_attributes("style='display:inline'").set_caption('First 5 rows of the dataset')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.850000,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.950000,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.850000,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.300000,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.700000,151.65,Yes


# 3. Initial Data Inspection
Check the first rows, data types, basic statistics, and missing values.


## Feature Types Summary

| Feature | Variable Type | Notes |
|--------|---------------|-------|
| `customerID` | Identifier (ID) | Unique identifier, excluded from analysis and modeling |
| `gender` | Categorical (binary, nominal) | Male / Female |
| `SeniorCitizen` | Binary (0/1, categorical) | Treated as categorical despite numeric encoding |
| `Partner` | Categorical (binary) | Yes / No |
| `Dependents` | Categorical (binary) | Yes / No |
| `tenure` | Numerical (discrete) | Number of months as a customer |
| `PhoneService` | Categorical (binary) | Yes / No |
| `MultipleLines` | Categorical (nominal) | Yes / No / No phone service |
| `InternetService` | Categorical (nominal) | DSL / Fiber optic / No |
| `OnlineSecurity` | Categorical (nominal) | Yes / No / No internet service |
| `OnlineBackup` | Categorical (nominal) | Yes / No / No internet service |
| `DeviceProtection` | Categorical (nominal) | Yes / No / No internet service |
| `TechSupport` | Categorical (nominal) | Yes / No / No internet service |
| `StreamingTV` | Categorical (nominal) | Yes / No / No internet service |
| `StreamingMovies` | Categorical (nominal) | Yes / No / No internet service |
| `Contract` | Categorical (ordinal) | Month-to-month < One year < Two year |
| `PaperlessBilling` | Categorical (binary) | Yes / No |
| `PaymentMethod` | Categorical (nominal) | Four different payment methods |
| `MonthlyCharges` | Numerical (continuous) | Monthly customer charges |
| `TotalCharges` | Numerical (continuous) | Total accumulated charges |
| `Churn` | Binary (target variable) | Customer churn indicator |



In [4]:
#Check for missing values
missing_values = df.isnull().sum()
print("Missing values in each column:")
print(missing_values)

#checking for duplicates in the dataset
df.duplicated().sum()

Missing values in each column:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


np.int64(0)

In [5]:
from src.eda.visualization import plot_eda_grid_plotly

features = df.columns.tolist()
plot_eda_grid_plotly(df, features, cols=4)
plot_eda_grid_plotly(df, features, target="Churn", cols=4)


## Resumen General
El dataset de Telco Customer Churn contiene información de 7043 clientes con 21 variables. La tasa de churn es del 26.5%.

## Conclusiones por Variable vs Churn

### Variables Demográficas
- **gender**: No muestra diferencias significativas en la tasa de churn entre hombres y mujeres. Ambos géneros tienen tasas similares (~26-27%).
- **SeniorCitizen**: Los clientes mayores de 65 años (SeniorCitizen=1) tienen una tasa de churn significativamente mayor (~42%) comparado con los no seniors (~23%). Es un factor de riesgo importante.
- **Partner**: Los clientes sin pareja (Partner=No) tienen mayor propensión a churn (~33%) que aquellos con pareja (~20%).
- **Dependents**: Los clientes sin dependientes (Dependents=No) churnan más (~31%) que aquellos con dependientes (~15%). Indica estabilidad familiar.

### Variables de Servicio
- **tenure**: Existe una relación inversa fuerte. Clientes con menor antigüedad (0-12 meses) tienen tasas de churn del ~50%, mientras que aquellos con >60 meses bajan al ~10%. Es una de las variables más predictivas.
- **PhoneService**: No hay diferencia significativa en churn entre clientes con y sin servicio telefónico.
- **MultipleLines**: Clientes con múltiples líneas tienen ligeramente mayor churn (~29%) que aquellos sin (~25%), pero la diferencia es marginal.

### Variables de Internet
- **InternetService**: Clientes con fibra óptica tienen la tasa de churn más alta (~42%), seguido de DSL (~19%) y sin internet (~7%). La fibra óptica parece problemática.
- **OnlineSecurity**: Clientes sin seguridad online churnan más (~42%) que aquellos con el servicio (~15%).
- **OnlineBackup**: Similar patrón; sin backup (~40% churn) vs con backup (~22%).
- **DeviceProtection**: Sin protección (~39% churn) vs con (~23%).
- **TechSupport**: Sin soporte técnico (~42% churn) vs con (~15%).
- **StreamingTV** y **StreamingMovies**: Clientes sin estos servicios tienen ligeramente mayor churn (~34% vs ~30%), pero no es tan fuerte como otros servicios adicionales.

### Variables Contractuales y de Pago
- **Contract**: 
  - Month-to-month: 43% churn (muy alto)
  - One year: 11%
  - Two year: 3%
  Los contratos a largo plazo reducen drásticamente el churn.
- **PaperlessBilling**: Clientes con facturación electrónica churnan más (~34%) que sin (~16%).
- **PaymentMethod**: 
  - Electronic check: 45% churn (más alto)
  - Mailed check: 19%
  - Bank transfer: 17%
  - Credit card: 15%
  Los métodos automáticos/electrónicos tienen mayor churn.
- **MonthlyCharges**: Relación positiva; clientes con cargos mensuales altos (>70) tienen mayor churn (~40%) que bajos (<30) (~10%).
- **TotalCharges**: Correlacionado con tenure; clientes con totales bajos churnan más, pero es menos directo que monthly charges.

## Valoración de Outliers
- **tenure**: Sin outliers significativos; rango 0-72 meses, distribución razonable.
- **MonthlyCharges**: Algunos valores altos (~118), pero no considerados outliers extremos. Distribución sesgada a la derecha.
- **TotalCharges**: 11 valores faltantes (espacios en blanco), tratados como missing. Sin outliers numéricos extremos.
- **Variables Categóricas**: Ninguna tiene categorías con frecuencias extremadamente bajas que puedan considerarse outliers.
- **General**: El dataset tiene pocos outliers numéricos. Los principales issues son los missing values en TotalCharges y posibles inconsistencias en categorías como "No internet service".

## Otras Observaciones
- **Correlaciones**: Tenure y TotalCharges están altamente correlacionadas (0.83). MonthlyCharges correlaciona moderadamente con varios servicios de internet.
- **Missing Values**: Solo TotalCharges tiene 11 missing (0.16%), probablemente clientes nuevos.
- **Duplicados**: No hay duplicados en el dataset.
- **Recomendaciones**: Enfocarse en retener clientes nuevos (bajo tenure), promover contratos largos, mejorar servicios de internet (especialmente fibra).